# **Simple Mean-Reverting Model**

## with and without Transaction Costs

In [1]:
import numpy as np
import pandas as pd
import yfinance as yf

In [74]:
startDate = "2006-01-01"
endDate = "2006-12-31"

tickers = ["AAPL", "MSFT", "GOOG", "AMZN"]

# Download data
df = yf.download(tickers, start=startDate, end=endDate)["Close"]

# Compute daily returns
dailyRet = df.pct_change()

# Cross-sectional mean return
marketDailyRet = dailyRet.mean(axis=1)

# Compute weights (mean-reversion signal)
weights = -(dailyRet.sub(marketDailyRet, axis=0))

# Normalize weights
wtsum = weights.abs().sum(axis=1)
weights = weights.div(wtsum, axis=0).fillna(0)

# Shift weights (no lookahead bias)
weights_shifted = weights.shift()

# Daily PnL
dailypnl = (weights_shifted * dailyRet).sum(axis=1)

# Drop NaNs
dailypnl = dailypnl.dropna()

# Sharpe ratio
sharpeRatio = np.sqrt(252) * dailypnl.mean() / dailypnl.std()

print("Sharpe Ratio:", sharpeRatio)

[*********************100%***********************]  4 of 4 completed


Sharpe Ratio: -0.1001705386771659


In [75]:
onewaytcost = 0.0005

# Turnover = change in weights
turnover = weights.diff().abs().sum(axis=1)

# Adjust PnL
dailypnl_tc = dailypnl - onewaytcost * turnover

# Sharpe ratio with cost
sharpeRatio_tc = np.sqrt(252) * dailypnl_tc.mean() / dailypnl_tc.std()

print("Sharpe (with cost):", sharpeRatio_tc)

Sharpe (with cost): -0.9611032808768589
